# 🎯 [QLT04_M01] 사단/여단별 주요 전술 장비(전차, 자주포, 장갑차) 일일 가동률 및 정비 대기 데이터셋 정의 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/QLT04/qlt04_m01_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 품질·규격·검사 (Level 2) |
| **미션 ID** | QLT04_M01 |
| **미션 주제** | 사단/여단별 주요 전술 장비(전차, 자주포, 장갑차) 일일 가동률 및 정비 대기 데이터셋 정의 |
| **소요 시간** | 약 1.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
외주정비는 정비 대기, 외주 검사 대기, 재작업, 규격·제원 불일치가 겹치면서 장비 가동률을 떨어뜨리지만 요인별 영향이 한눈에 보이지 않아 어디에 정비 자원을 먼저 투입할지 정하기 어렵다. 본 프로젝트는 미 국방부 외주정비 품질보증 사례의 문제의식(규격 정합성·검사 병목·하자 연결)을 교육용 가상 자료로 재구성해, 정비 대기 요인 분류, 저가동 요인 분석, 재배치 시뮬레이션, 추세 대시보드, 사람 검토 워크플로우까지 프로토타입으로 구현하는 과제이다. 가상 장비는 유형 A·B·C와 가상 부대로만 구분하며 실제 장비·부대 자료는 쓰지 않는다. 최종 목표는 가동률 저하 요인과 외주 품질검사 병목을 근거와 함께 보여 주되, 정비 우선순위와 자원 투입 결정은 정비관·품질검사관·군수참모가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 프로세스 정의서(`qlt04_as_is_to_be_spec.md`), UAT 피드백 리포트(`qlt04_uat_feedback_report.md`), 운영 매뉴얼(`qlt04_final_manual.md`) 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 품질 판정 룰 엔진, 릿지 회귀 XAI 엔진, 시뮬레이터 UI, HITL 결재 FSM, 배치 러너 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 야전 배포를 위한 최종 패키지(`qlt04_package/`)

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 사단/여단별 주요 전술 장비(전차, 자주포, 장갑차) 일일 가동률 및 정비 대기 데이터셋 정의
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

전술 장비(전차, 자주포) 일일 가동률 및 정비 대기 데이터셋을 정의한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **부대별 전술 장비(유형 A·B·C로 추상화한 가상 장비)의 일일 가동률과 정비 대기 자료의 구조와 입출력 명세를 정의하고, 외주정비 입고부터 품질검사·가동 복귀까지의 현행(AS-IS) 업무를 단계로 나누어 병목을 찾는다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **장비 식별, 정비 요청, 대기 요인별 일수(부품·정비병·외주 정비·외주 검사), 품질 지표(재작업·규격 불일치·하자), 자원 수준, 가동률을 묶어 필드 16개의 스키마와 검증 규칙 작성하여 원인을 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/qlt04_as_is_to_be_spec.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 현행 업무(AS-IS)와 개선 업무(TO-BE) 비교의 명확성

#### 미션 PASS 조건

- 정량 기준: 업무 단계 분해 10단계 이상, 병목 구간 3개소 이상 도출


- 진행 기준: 업무 정의서 및 입출력 데이터 명세 작성 완료


- 검증 방법: 실무 프로세스와의 일치성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""가상 장비 가동률·정비 대기 자료(필드 16개)의 스키마와 검증 규칙을 정의하고 오류 주입 시험으로 검출력을 확인한 뒤, 정비 입고부터 외주 정비 검사·가동 복귀까지의 현행(AS-IS) 업무를 12단계로 분해해 병목 구간과 개선(TO-BE) 효과를 비교한다."""
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 장비 가동률·정비 대기 자료 생성기 (실제 부대·장비·제원·가동률 정보 없음)
# 장비 유형 A(궤도형)·B(화포형)·C(차륜형)은 추상 구분이며 실제 장비와 무관하다.
# 최근 90일 동안 장비 1대의 정비 대기 요인(부품·정비병·외주 정비·외주 검사·재작업)과 가동률을 만든다.
# ---------------------------------------------------------------------------
UNITS = ["가상A부대", "가상B부대", "가상C부대", "가상D부대", "가상E부대", "가상F부대"]
ETYPES = ["유형A", "유형B", "유형C"]
WINDOW = 90                                       # 관측 기간(일)
CAUSES = ["부품 대기", "정비병 부족", "외주 정비", "검사·재작업"]
CAUSE_COL = {"부품 대기": "parts_wait_days", "정비병 부족": "labor_wait_days", "외주 정비": "outsource_days", "검사·재작업": "inspect_wait_days"}
NUM_FIELDS = ["age_years", "op_hours", "maint_orders", "parts_wait_days", "labor_wait_days", "outsource_days", "inspect_wait_days", "rework_count",
              "spec_mismatch", "defect_reports", "parts_stock_pct", "mechanic_fill_pct", "avail_pct"]
FEAT = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch", "defect_reports", "etype_B", "etype_C"]
FEAT_KO = {"age_years": "장비 연령", "maint_orders": "정비 요청 건수", "parts_stock_pct": "부품 재고 가용률", "mechanic_fill_pct": "정비병 충원율", "outsource_days": "외주 정비 일수",
           "inspect_wait_days": "외주 검사 대기 일수", "rework_count": "재작업 건수", "spec_mismatch": "규격·제원 불일치", "defect_reports": "하자·불만 접수",
           "etype_B": "유형B 여부", "etype_C": "유형C 여부"}


def unit_profile(seed=3):
    """부대별 고정 특성: 부품 재고 수준, 정비병 충원율, 외주 정비소 거리(일수 가중), 외주 검사 혼잡도."""
    r = np.random.default_rng(seed)
    return pd.DataFrame({"unit": UNITS, "stock": r.uniform(45, 95, len(UNITS)), "fill": r.uniform(55, 98, len(UNITS)),
                         "dist": r.uniform(0.8, 1.6, len(UNITS)), "queue": r.uniform(0.7, 2.2, len(UNITS))})


def make_fleet(n=300, seed=7, shock=0.0):
    """장비 n대의 90일 관측 자료(필드 16개)와 정답 요인을 만든다. shock은 외주 검사 병목 악화 정도(AAR용 환경 변화)."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    ui = rng.integers(0, len(UNITS), n)
    et = rng.choice(ETYPES, n, p=[.35, .3, .35])
    age = np.clip(rng.gamma(4, 2.2, n), 0.5, 30).round(1)
    eff = np.array([{"유형A": 1.15, "유형B": 1.0, "유형C": 0.9}[t] for t in et])
    orders = rng.poisson(np.clip(1.2 + 0.12 * age, 0.5, 6) * eff)
    stock = np.clip(pf["stock"].to_numpy()[ui] + rng.normal(0, 8, n), 20, 100).round(0)
    fill = np.clip(pf["fill"].to_numpy()[ui] + rng.normal(0, 6, n), 30, 100).round(0)
    parts_w = (orders * rng.gamma(2, 0.55 * (1 + ((100 - stock) / 28) ** 2) / 2)).round(0)
    labor_w = (orders * rng.gamma(2, 0.7 * (1 + ((100 - fill) / 24) ** 2) / 2)).round(0)
    n_out = rng.binomial(orders, 0.3)
    out_d = (n_out * rng.gamma(3, 4.5 * pf["dist"].to_numpy()[ui] / 3)).round(0)
    insp_w = (n_out * rng.gamma(2, (2.6 + shock) * pf["queue"].to_numpy()[ui] / 2)).round(0)
    spec = rng.poisson(0.2 + 0.04 * age)
    rework = rng.binomial(n_out, np.clip(0.08 + 0.1 * spec, 0, .6))
    defect = rng.poisson(0.3 + 0.7 * rework)
    base_rep = orders * 1.8
    down = base_rep + parts_w + labor_w + out_d + insp_w + rework * 4 + rng.normal(0, 1.2, n)
    comp = np.column_stack([parts_w, labor_w, out_d, insp_w + rework * 4])
    df = pd.DataFrame({"equip_id": [f"E-{i + 1:04d}" for i in range(n)], "unit": np.array(UNITS)[ui], "etype": et, "age_years": age,
                       "op_hours": np.clip(rng.normal(320 - 4 * age, 70, n), 40, 600).round(0), "maint_orders": orders,
                       "parts_wait_days": parts_w, "labor_wait_days": labor_w, "outsource_days": out_d, "inspect_wait_days": insp_w,
                       "rework_count": rework, "spec_mismatch": spec, "defect_reports": defect, "parts_stock_pct": stock, "mechanic_fill_pct": fill,
                       "avail_pct": (100 * (1 - np.clip(down, 0, WINDOW) / WINDOW)).round(1)})
    df["main_cause"] = np.where(comp.max(axis=1) < 3, "없음", np.array(CAUSES)[comp.argmax(axis=1)])
    return df


def feat_frame(df):
    """회귀·분석용 설명 변수 표(유형은 더미 변수)."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    return x[FEAT]


def make_trend(months=12, seed=5, shock_unit=None):
    """[교육용가상자료] 부대 x 유형 x 월 평균 가동률 추세. shock_unit 부대는 하반기에 외주 검사 병목으로 가동률이 떨어진다."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    rows = []
    for _, u in pf.iterrows():
        base = 62 + 0.22 * u["stock"] + 0.12 * u["fill"] - 3.0 * u["queue"]
        for t in ETYPES:
            off = {"유형A": -3.0, "유형B": 0.0, "유형C": 2.0}[t]
            for m in range(months):
                v = base + off + 2.0 * np.sin(m / 12 * 2 * np.pi) + rng.normal(0, 1.0)
                if shock_unit == u["unit"] and m >= months // 2:
                    v -= 1.5 * (m - months // 2 + 1)
                rows.append((u["unit"], t, m + 1, round(float(np.clip(v, 30, 99)), 1)))
    return pd.DataFrame(rows, columns=["unit", "etype", "month", "avail_pct"])

# 1) 스키마 정의: 장비 식별, 정비 요청, 대기 요인별 일수(부품·정비병·외주 정비·외주 검사), 품질 지표(재작업·규격 불일치·하자), 자원 수준, 가동률
SCHEMA = {
    "equip_id": {"type": str, "pattern": r"^E-\d{4}$", "desc": "장비 번호"},
    "unit": {"type": str, "enum": UNITS, "desc": "소속 부대(가상)"},
    "etype": {"type": str, "enum": ETYPES, "desc": "장비 유형(추상 구분)"},
    "age_years": {"type": (int, float), "min": 0, "max": 40, "desc": "장비 연령(년)"},
    "op_hours": {"type": (int, float), "min": 0, "max": 2160, "desc": "최근 90일 가동 시간"},
    "maint_orders": {"type": int, "min": 0, "max": 30, "desc": "정비 요청 건수"},
    "parts_wait_days": {"type": (int, float), "min": 0, "max": 90, "desc": "부품 대기 일수"},
    "labor_wait_days": {"type": (int, float), "min": 0, "max": 90, "desc": "정비병 부족 대기 일수"},
    "outsource_days": {"type": (int, float), "min": 0, "max": 90, "desc": "외주 정비 일수"},
    "inspect_wait_days": {"type": (int, float), "min": 0, "max": 90, "desc": "외주 검사 대기 일수"},
    "rework_count": {"type": int, "min": 0, "max": 30, "desc": "재작업(검사 불합격) 건수"},
    "spec_mismatch": {"type": int, "min": 0, "max": 20, "desc": "규격·제원 불일치 건수"},
    "defect_reports": {"type": int, "min": 0, "max": 50, "desc": "하자·불만 접수 건수"},
    "parts_stock_pct": {"type": (int, float), "min": 0, "max": 100, "desc": "부품 재고 가용률(%)"},
    "mechanic_fill_pct": {"type": (int, float), "min": 0, "max": 100, "desc": "정비병 충원율(%)"},
    "avail_pct": {"type": (int, float), "min": 0, "max": 100, "desc": "가동률(%)"},
}


def check(name, v, r):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return [f"{name}: 필수 값 누락"]
    if not isinstance(v, r["type"]) or isinstance(v, bool):
        return [f"{name}: 자료형 오류({type(v).__name__})"]
    errs = []
    if "enum" in r and v not in r["enum"]:
        errs.append(f"{name}: 허용되지 않은 값 '{v}'")
    if "min" in r and v < r["min"]:
        errs.append(f"{name}: 최소값 {r['min']} 미만")
    if "max" in r and v > r["max"]:
        errs.append(f"{name}: 최대값 {r['max']} 초과")
    if "pattern" in r and not re.match(r["pattern"], v):
        errs.append(f"{name}: 형식 불일치 '{v}'")
    return errs


def validate(rec):
    """
    [TODO : 학생 실습 1] 단일 장비 레코드의 스키마 및 논리 제약조건 검증 로직을 완성하세요.
    - 힌트 1: SCHEMA에 정의된 필수 필드가 rec에 모두 포함되어 있는지 확인합니다 (누락 시 f"{k}: 필드 누락").
    - 힌트 2: 각 필드에 대해 check(k, rec[k], r) 함수를 호출하여 자료형/범위 오류를 수집합니다.
    - 힌트 3: 필드 간 논리 규칙을 점검합니다:
      * rework_count > maint_orders: "rework_count: 정비 요청 건수보다 재작업 건수가 많음"
      * maint_orders == 0 이고 대기일수 합계 > 0: "maint_orders: 정비 요청이 없는데 대기 일수가 있음"
      * 대기일수 합계 > 95: "wait_days: 대기 일수 합계가 관측 기간(90일)을 초과"
    - 반환값: 오류 메시지 문자열 리스트 (오류가 없으면 빈 리스트 [])
    """
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.


def to_record(row):
    return {k: (row[k].item() if hasattr(row[k], "item") else row[k]) for k in SCHEMA}


fleet = make_fleet(100, seed=7)
records = [to_record(r) for _, r in fleet.iterrows()]
ok = sum(not validate(r) for r in records)
print(f"[스키마 검증] 장비 {len(records)}대 중 통과 {ok}대, 오류 {len(records) - ok}대 / 필드 {len(SCHEMA)}개")
print(f"평균 가동률 {fleet['avail_pct'].mean():.1f}%, 가동률 70% 미만 {int((fleet['avail_pct'] < 70).sum())}대, 외주 정비 경험 장비 {int((fleet['outsource_days'] > 0).sum())}대, 규격·제원 불일치 1건 이상 {int((fleet['spec_mismatch'] > 0).sum())}대")

# 2) 오류 주입 시험: 형식·허용값·범위·자료형·누락·논리 규칙 위반
bad = dict(records[2], equip_id="E12", unit="미상부대", age_years=-3, op_hours="많음")
bad2 = dict(records[5], maint_orders=2, rework_count=5)
bad3 = dict(records[0], maint_orders=0, parts_wait_days=5)
bad4 = {k: v for k, v in records[7].items() if k != "avail_pct"}
bad5 = dict(records[9], avail_pct=None)
print("\n[오류 주입 결과]")
for e in validate(bad) + validate(bad2) + validate(bad3) + validate(bad4) + validate(bad5):
    print(" -", e)

# 3) AS-IS 업무 단계 분해: [교육용가상자료] 외주 정비 1건의 단계별 소요일(작업 + 대기)
rng = np.random.default_rng(3)
STEPS = [("정비 요청 접수·고장 접수", 0.5, 1), ("고장 진단·정비 계획 수립", 1, 1), ("부품 청구·조달", 1, 6), ("정비병 배정", 0.5, 3),
         ("부대 정비 수행", 2, 1), ("외주 정비 의뢰·입고", 1, 4), ("외주 정비 수행", 6, 2), ("외주 검사 대기", 0, 5),
         ("외주 정비 품질검사(규격·제원 대조)", 1, 1), ("불합격 시 재작업·하자 조치", 3, 3), ("출고 승인·부대 인수", 0.5, 2), ("가동 복귀 확인·이력 기록", 0.5, 1)]
TOBE = {2: ("부품 소요 예측·사전 청구(미션 4)", 0.4), 3: ("정비병 부하 기반 배정 추천(미션 5)", 0.5), 7: ("검사 일정 자동 조율(미션 5)", 0.4),
        8: ("규격·제원 자동 대조 사전 점검(미션 3)", 0.5), 9: ("재작업 원인 자동 분류(미션 2·4)", 0.6), 11: ("이력 자동 기록(미션 2)", 0.3)}
rows = []
for i, (name, work, wait) in enumerate(STEPS):
    w = rng.gamma(4, max(work, 0.01) / 4, 30); q = rng.gamma(2, wait / 2, 30)
    f = TOBE.get(i, ("유지(사람 판단)", 1.0))
    rows.append({"단계": f"{i + 1:02d}. {name}", "작업(일)": w.mean() if work else 0.0, "대기(일)": q.mean(), "AS-IS(일)": (w.mean() if work else 0.0) + q.mean(),
                 "TO-BE 방식": f[0], "TO-BE(일)": ((w.mean() if work else 0.0) + q.mean()) * f[1]})
proc = pd.DataFrame(rows)
proc["병목"] = np.where(proc["대기(일)"] >= proc["대기(일)"].sort_values().iloc[-4], "병목", "")           # 대기일 상위 4개 단계를 병목으로 본다
print("\n[AS-IS / TO-BE 업무 단계 비교]")
print(proc.round(1).to_string(index=False))
n_bn = int((proc["병목"] == "병목").sum())
asis, tobe = proc["AS-IS(일)"].sum(), proc["TO-BE(일)"].sum()
print(f"\n업무 단계 {len(proc)}개(목표 10개 이상: {'달성' if len(proc) >= 10 else '미달'}), 병목 {n_bn}개소(목표 3개소 이상: {'달성' if n_bn >= 3 else '미달'})")
print(f"외주 정비 1건 소요: AS-IS {asis:.1f}일 -> TO-BE {tobe:.1f}일 ({1 - tobe / asis:.0%} 단축 예상). 품질검사 판정, 정비 우선순위 결정, 출고 승인은 사람이 수행한다.")

# 4) 입출력 데이터 명세 요약
print("\n[입출력 명세] 입력: 장비 가동·정비 대기 레코드(위 16개 필드) / 출력: 저가동 요인 순위, 저가동 장비 목록, 정비 자원 재배치 안, 검토·결재 이력")
print("[권한별 요구] 정비 담당: 입력·상신 / 정비관·품질검사관: 검토 / 군수참모: 정책 결정(미션 7에서 구현)")

plt.figure(figsize=(9, 5))
y = np.arange(len(proc))
plt.barh(y - 0.2, proc["AS-IS(일)"], 0.4, label="AS-IS", color="tab:gray")
plt.barh(y + 0.2, proc["TO-BE(일)"], 0.4, label="TO-BE", color="tab:blue")
plt.yticks(y, [f"step {i + 1}{' *' if b else ''}" for i, b in enumerate(proc["병목"])])
plt.gca().invert_yaxis(); plt.xlabel("days per case"); plt.legend()
plt.title("Outsourced maintenance flow: AS-IS vs TO-BE (* = bottleneck, synthetic)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
[스키마 검증] 장비 100대 중 통과 100대, 오류 0대 / 필드 16개
평균 가동률 79.3%, 가동률 70% 미만 25대, 외주 정비 경험 장비 50대, 규격·제원 불일치 1건 이상 49대

[오류 주입 결과]
 - equip_id: 형식 불일치 'E12'
 - unit: 허용되지 않은 값 '미상부대'
 - age_years: 최소값 0 미만
 - op_hours: 자료형 오류(str)
 - rework_count: 정비 요청 건수보다 재작업 건수가 많음
 - maint_orders: 정비 요청이 없는데 대기 일수가 있음
 - avail_pct: 필드 누락
 - avail_pct: 필수 값 누락

[AS-IS / TO-BE 업무 단계 비교]
                      단계  작업(일)  대기(일)  AS-IS(일)                TO-BE 방식  TO-BE(일) 병목
      01. 정비 요청 접수·고장 접수    0.5    1.0       1.5               유지(사람 판단)       1.5   
      02. 고장 진단·정비 계획 수립    1.3    0.9       2.2               유지(사람 판단)       2.2   
            03. 부품 청구·조달    1.0    6.6       7.7    부품 소요 예측·사전 청구(미션 4)       3.1 병목
              04. 정비병 배정    0.6    3.0       3.6   정비병 부하 기반 배정 추천(미션 5)       1.8   
            05. 부대 정비 수행    2.4    1.1       3.5               유지(사람 판단)       3.5   
         06. 외주 정비 의뢰·입고    1.0    4.1       5.1               유지(사람 판단)       5.1 병목
            07. 외주 정비 수행    5.3    1.9       7.2               유지(사람 판단)       7.2   
            08. 외주 검사 대기    0.0    4.9       4.9       검사 일정 자동 조율(미션 5)       2.0 병목
09. 외주 정비 품질검사(규격·제원 대조)    1.1    1.0       2.1 규격·제원 자동 대조 사전 점검(미션 3)       1.0   
     10. 불합격 시 재작업·하자 조치    3.1    3.6       6.7    재작업 원인 자동 분류(미션 2·4)       4.0 병목
         11. 출고 승인·부대 인수    0.5    2.3       2.7               유지(사람 판단)       2.7   
      12. 가동 복귀 확인·이력 기록    0.5    1.0       1.6          이력 자동 기록(미션 2)       0.5   

업무 단계 12개(목표 10개 이상: 달성), 병목 4개소(목표 3개소 이상: 달성)
외주 정비 1건 소요: AS-IS 48.9일 -> TO-BE 34.7일 (29% 단축 예상). 품질검사 판정, 정비 우선순위 결정, 출고 승인은 사람이 수행한다.

[입출력 명세] 입력: 장비 가동·정비 대기 레코드(위 16개 필드) / 출력: 저가동 요인 순위, 저가동 장비 목록, 정비 자원 재배치 안, 검토·결재 이력
[권한별 요구] 정비 담당: 입력·상신 / 정비관·품질검사관: 검토 / 군수참모: 정책 결정(미션 7에서 구현)
```
</details>
